# Prompt Inference via `swift deploy`

这个 notebook 用来连接已经启动的 `swift deploy` 服务，而不是在 notebook 进程里直接加载模型。

先在一个 terminal 里启动服务：

```bash
bash /user-data/yifengx4/DeepAnalyze/scripts/launch_swift_deploy_agent_bw.sh
```

服务启动后会提供 OpenAI-compatible API：`http://127.0.0.1:8000/v1`。

和原来的 `infer_prompt_swift_agent_bw.ipynb` 一样，这里默认仍然把完整上下文放在一个 `user` message 里：

```python
messages = [{'role': 'user', 'content': PROMPT}]
```

也就是说，后续测试只需要复用原来的 prompt/chat message 内容，把后端从 `SwiftInfer(...)` 换成 HTTP client。

In [1]:
from openai import OpenAI

BASE_URL = 'http://127.0.0.1:8000/v1'
MODEL = 'LocalLLM'
API_KEY = 'EMPTY'  # If launch script is given API_KEY=..., use the same value here.

client = OpenAI(api_key=API_KEY, base_url=BASE_URL)

print(f'BASE_URL = {BASE_URL}')
print(f'MODEL = {MODEL}')

BASE_URL = http://127.0.0.1:8000/v1
MODEL = LocalLLM


## Check The Service

如果下面这个 cell 报 connection error，说明 `swift deploy` 服务还没有启动，或者端口/机器地址不对。

In [2]:
models = client.models.list()
print('served models:')
for model in models.data:
    print(f'  {model.id}')

served models:
  DeepAnalyze-8B


## Client Helpers

`run_prompt()` 对齐原 notebook 的入口：传入一个大字符串 prompt，内部包装成 `messages = [{'role': 'user', 'content': prompt}]`。

`run_messages()` 用于后续多轮测试，直接传 OpenAI chat messages。

In [3]:
def _print_message(message):
    print(message.content or '')
    tool_calls = getattr(message, 'tool_calls', None)
    if tool_calls:
        print('\nTool calls:')
        for tool_call in tool_calls:
            print(tool_call.function)


def run_messages(
    messages: list[dict],
    *,
    tools: list[dict] | None = None,
    temperature: float = 0,
    max_tokens: int = 16428,
    stream: bool = False,
):
    kwargs = {
        'model': MODEL,
        'messages': messages,
        'temperature': temperature,
        'max_tokens': max_tokens,
        'stream': stream,
    }
    if tools is not None:
        kwargs['tools'] = tools

    if stream:
        chunks = client.chat.completions.create(**kwargs)
        text = ''
        for chunk in chunks:
            delta = chunk.choices[0].delta.content or ''
            text += delta
            print(delta, end='', flush=True)
        print()
        return text

    response = client.chat.completions.create(**kwargs)
    message = response.choices[0].message
    _print_message(message)
    return message


def run_prompt(
    prompt: str,
    *,
    system: str | None = None,
    tools: list[dict] | None = None,
    temperature: float = 0,
    max_tokens: int = 16428,
    stream: bool = False,
):
    messages = []
    if system:
        messages.append({'role': 'system', 'content': system})
    messages.append({'role': 'user', 'content': prompt})
    return run_messages(
        messages,
        tools=tools,
        temperature=temperature,
        max_tokens=max_tokens,
        stream=stream,
    )

## Optional Tool Schema

原 notebook 的 prompt 已经把工具说明写进了 `PROMPT` 文本里，所以多数情况下可以不传 `tools`。如果后续想走 OpenAI-style tool schema，可以把下面的 `DATAFLOW_TOOLS` 传给 `run_prompt(..., tools=DATAFLOW_TOOLS)`。

In [4]:
DATAFLOW_TOOLS = [
    {
        'type': 'function',
        'function': {
            'name': 'createOrModifyOperator',
            'description': 'Add or modify a Python function as an operator in the dataflow, then automatically execute it.',
            'parameters': {
                'type': 'object',
                'properties': {
                    'operatorId': {'type': 'string', 'description': 'Unique operator name.'},
                    'code': {'type': 'string', 'description': 'Python function: def load() or def process(...).'},
                    'summary': {'type': 'string', 'description': 'Detailed summary of the operator behavior.'},
                },
                'required': ['operatorId', 'code'],
            },
        },
    },
    {
        'type': 'function',
        'function': {
            'name': 'deleteOperator',
            'description': 'Delete an operator from the workflow, including connected links.',
            'parameters': {
                'type': 'object',
                'properties': {
                    'operatorId': {'type': 'string', 'description': 'ID of the operator to delete.'},
                },
                'required': ['operatorId'],
            },
        },
    },
    {
        'type': 'function',
        'function': {
            'name': 'executeOperator',
            'description': "Execute the workflow and get the specified operator's result.",
            'parameters': {
                'type': 'object',
                'properties': {
                    'operatorId': {'type': 'string', 'description': 'The operator ID to view result for.'},
                },
                'required': ['operatorId'],
            },
        },
    },
]

## Prompt Copied From The Original Notebook

下面这个 cell 严格复用 `infer_prompt_swift_agent_bw.ipynb` 里的完整 `PROMPT` 和调用方式：

```python
message = run_prompt(PROMPT)
```

区别只有后端：这里的 `run_prompt()` 通过 `swift deploy` 的 OpenAI-compatible API 调模型。

In [6]:
# Edit this cell and run it to see the model output.
PROMPT = '''
<｜begin▁of▁sentence｜>You are a data science Copilot that helps users solve data-centric tasks by building dataflows.

## What is Dataflow?

Dataflow represents data analysis as a DAG (directed acyclic graph) where:
- Each **operator** is a single step of data processing
- Each **link** represents data dependency between operators
- Each operator receives table(s) from input operator(s), processes them, and outputs a single table
- The output table can be viewed via execution, or passed to downstream operators via links

## Context Format

Your conversation context is a single message with three top-level sections, in this order:

- `# Completed Tasks` — previous tasks you've already finished (omitted if none)
- `# Ongoing Task` — the current task, including turns you've taken so far
- `# Current Dataflow` — the live DAG: every operator's current state

**Overall layout:**

```
# Completed Tasks

## Task (completed)

### User request

<a past user question>

### Turn 1
Thought: <your reasoning from that turn>
- <toolName> (succeeded)
  - Summary: <the summary you provided in the tool call>
  - Output: <brief tool output>

## Task (completed)

### User request

<another past user question>

### Turn 1
...

# Ongoing Task
## Task (ongoing)

### User request

<the current user question>

### Turn 1
Thought: ...
- <toolName> (succeeded)
  - Summary: ...
  - Output: ...

### Turn 2
Thought: ...
- <toolName> (failed)
  - Summary: ...
  - Error:
    <full error trace, possibly multi-line>
- <toolName> (succeeded)
  - Summary: ...
  - Output: ...
  - Execution Error:
    <runtime error — shown here only if the operator has since been resolved or replaced. If the operator is still errored in the Current Dataflow below, this sub-bullet (and Summary) is omitted because you can read the live error there.>
- <toolName> (succeeded)
  - Summary: ...
  - Output: ...

# Current Dataflow
## Operators

### Operator `<operator_id>` (<operator_type>, executed|failed|not-executed)
Summary: <what the operator does>
Properties:
  code: <the operator's code (when available)>
Result:
  <execution output, table shape, and sample data>

### Operator `<another_operator_id>` ...
...

## Links
- <source_id> → <target_id>
```

## Key Principles

- **Call tools only through the native protocol**: Invoke tools using the tool-call mechanism. Never emit `<action>`, `<thought>`, `<operator>`, or any other tag-like structures in your response — those shapes appear in your input to describe past turns and existing state, never in your output.
- **One operation per operator**: Each operator does one task (join, filter, aggregate, etc.). Use links to connect them.
- **Build incrementally**: Link new operators to existing ones. Never recreate data already in the workflow.
- **Read documentation first**: When the task mentions abstract concepts, load documentation to understand exact definitions.
- **Write detailed, parameter-rich summaries**: Every operator's summary must enumerate the key parameters of what it does. For **data loading**, include the filename (not the full path) and any non-default parameters — skip rows, header setting, delimiter, encoding, row limit (e.g., `nrows=5`). For **data processing**, include the column names involved, join keys and join type, filter conditions with thresholds, groupby keys, and aggregation methods. A summary like "Load customers.csv" or "Process data" is not acceptable — a reader who sees only the summary should understand the operator's intent.
- **Refine or fix operator in place by modifying operators**: When an operator errors or produces an unexpected result, modify that operator directly — don't add a downstream operator to patch the output or recreate the pipeline. For execution errors, read the error message and the input operator's result, then rewrite the failing operator's code. For semantically wrong results, trace back to the operator whose logic is off (often upstream of where you first noticed the problem) and fix it in place.
- **Debug by isolating**: When encountering unexpected results, isolate the problematic logic into its own operator.
- **Understand column semantics**: Before analysis, examine column names and their stats to understand what each column represents. Columns may carry semantic meaning that affects how data should be filtered or interpreted — respect these signals and apply appropriate preprocessing before computing results.
- **Load all data before subsetting**: When the question requires comparing across groups, load all relevant files first, then determine the correct subset.
- **Handle messy data files**: Load data files directly in a single operator. Real-world data files are often malformed — they may have wrong delimiters, missing or misplaced headers, metadata/comment rows, or multiple tables in one file. After loading, inspect the result. If column names look auto-generated (e.g., `Unnamed: 0`) or a data value appears as a header, adjust the loading parameters (e.g., `header=`, `skiprows=`, `sep=`) by modifying the data loading operator.
- **Avoid monolithic code blocks**: Do NOT write one large operator that does everything — you cannot tell which step failed, inspect intermediate results, or debug without re-running everything. Instead, decompose into separate operators each doing ONE thing (e.g., filter → join → aggregate → filter → join → final filter). Each can be executed and verified independently.

Answer the following questions as best you can. You have access to the following tools:

createOrModifyOperator: Call this tool to interact with the createOrModifyOperator API. What is the createOrModifyOperator API useful for? Add or modify a Python function as an operator in the dataflow, then automatically execute it.
- If operatorId does NOT exist: creates a new operator
- If operatorId exists: modifies the existing operator (must keep same type)
- After creation/modification, the operator is automatically executed.

RULES:
1. operatorId must be a valid Python variable name
2. Function name MUST be exactly "load" or "process"
3. For process(): each parameter MUST match an existing operator's operatorId - links are auto-created/updated

## def load() -> pd.DataFrame
Load data from files. No input parameters. File I/O allowed.

Example: operatorId="customers"
  def load() -> pd.DataFrame:
      return pd.read_csv('/data/customers.csv')

## def process(opId1, opId2, ...) -> pd.DataFrame
Transform input data. Each parameter references an existing operator id. Links between operators will be auto-created.

Example: operatorId="filtered" (requires "customers" to exist)
  def process(customers) -> pd.DataFrame:
      return customers[customers['age'] > 18] Parameters: {"type": "object", "properties": {"operatorId": {"type": "string", "description": "Unique operator name (valid Python variable). Other operators reference this as input parameter."}, "code": {"type": "string", "description": "Python function: def load() or def process(...)"}, "summary": {"type": "string", "description": "Detailed summary of the operator behavior. For load() operators, include the filename(s) being loaded."}}, "required": ["operatorId", "code"]} Format the arguments as a JSON object.

deleteOperator: Call this tool to interact with the deleteOperator API. What is the deleteOperator API useful for? Delete an operator from the workflow. This also deletes all connected links. Parameters: {"type": "object", "properties": {"operatorId": {"type": "string", "description": "ID of the operator to delete"}}, "required": ["operatorId"]} Format the arguments as a JSON object.

executeOperator: Call this tool to interact with the executeOperator API. What is the executeOperator API useful for? Execute the workflow and get the specified operator's result. The execution result (if succeeded) includes the shape of the input tables (if any) and output table, and the records in the output table. Parameters: {"type": "object", "properties": {"operatorId": {"type": "string", "description": "The operator ID to view result for."}}, "required": ["operatorId"]} Format the arguments as a JSON object.

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [createOrModifyOperator,deleteOperator,executeOperator]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can be repeated zero or more times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!
<｜User｜>
# Ongoing Task
## Task (ongoing)

### User request

You are a data scientist. Answer the following question based on the data files.

Data files available (use below relative paths to read the data):
[
  "data/legal/input/csn-data-book-2024-csv/CSVs/2024_CSN_Data_Contributors.csv"
]

Note: All paths are relative. Some paths may contain wildcards (e.g., "folder/*" or "file-*.csv"). Use glob patterns to match and read those files.

Question: How many frauds were reported by FTC over the web between 2022 and 2024 in total?

Your last line MUST BE: **Final Answer: <value>**

### Turn 1
- createOrModifyOperator(operatorId="contributors") (succeeded)

### Turn 2
- createOrModifyOperator(operatorId="contributors") (succeeded)

### Turn 3
- createOrModifyOperator(operatorId="clean_ftc") (succeeded)

### Turn 4
- createOrModifyOperator(operatorId="contributors") (succeeded)

### Turn 5
- createOrModifyOperator(operatorId="ftc_fraud_web") (succeeded)

### Turn 6
- createOrModifyOperator(operatorId="total_fraud_web") (succeeded)

### Turn 7
- executeOperator(operatorId="total_fraud_web") (succeeded)

Above is user's request and the steps you already took. You as an assistant please keep working on solving user's request based on the progress of current workflow.

# Current Dataflow
## Operators

### Operator `contributors` (DataLoading, executed)
Summary: Load contributors.csv: Load the CSN Data Contributors CSV file, skipping the first 2 rows to get the correct header and data; set column names to Year, Data Contributor, # of Reports, %
Properties:
  code: def load() -> pd.DataFrame:
    df = pd.read_csv('data/legal/input/csn-data-book-2024-csv/CSVs/2024_CSN_Data_Contributors.csv', skiprows=2)
    # Set proper column names based on the actual headers
    df.columns = ['Year', 'Data Contributor', '# of Reports', '%']
    return df
Result:
  Executed operator contributors
  Output table shape: (140, 4)
    FTC  Unnamed: 1  Unnamed: 2  Unnamed: 3
  0  Year  Data Contributor  # of Reports  %
  1  2022  FTC - Web Reports (IDT)  796,366  14.98%
  2  2022  FTC - Web Reports (Fraud & Other)  693,789  13.05%
  3  2022  FTC - Mobile Reports (Fraud & Other)  622,035  11.70%
  4  2022  FTC - Mobile Reports (IDT)  191,494  3.60%
  5  2022  FTC - "877 FTC HELP" (Fraud & Other)  155,803  2.93%
  6  2022  FTC - "877 ID THEFT"  100,294  1.89%
  7  2023  FTC - Web Reports (IDT)  696,163  12.55%
  ...  ...  ...  ...  ...
  133  2024  Hawaii, Office of Consumer Protection  667  0.01%
  134  2024  Alaska, Attorney General  329  0.01%
  135  2024  Nevada, Attorney General  294  0.00%
  136  NaN  NaN  NaN  NaN
  137  Percentages are based on the total number of Sentinel reports in 2022 (5,317,751), 2023 (5,548,815),...  NaN  NaN  NaN
  138  NaN  NaN  NaN  NaN
  139  Source: Consumer Sentinel Network Data Book 2024, Federal Trade Commission  NaN  NaN  NaN

### Operator `clean_ftc` (DataProcessing, executed)
Summary: Clean FTC data: Remove rows with NaN or text in Year, Data Contributor, or # of Reports columns; filter for rows containing "FTC" in Data Contributor column; remove commas from # of Reports column to enable numeric operations
Properties:
  code: def process(contributors) -> pd.DataFrame:
    # Remove rows with NaN or text in Year column
    df = contributors[contributors['Year'].notna() & contributors['Year'].astype(str).str.isnumeric()]
    # Remove rows with NaN or text in Data Contributor column
    df = df[df['Data Contributor'].notna() & df['Data Contributor'].astype(str).str.contains('FTC')]
    # Remove rows with NaN or text in # of Reports column
    df = df[df['# of Reports'].notna() & df['# of Reports'].astype(str).str.replace(',', '').str.isnumeric()]
    return df
Result:
  Executed operator clean_ftc
  Input operator(table shape): contributors(140, undefined)
  Output table shape: (18, 4)
    Year  Data Contributor  # of Reports  %
  0  2022  FTC - Web Reports (IDT)  796,366  14.98%
  1  2022  FTC - Web Reports (Fraud & Other)  693,789  13.05%
  2  2022  FTC - Mobile Reports (Fraud & Other)  622,035  11.70%
  3  2022  FTC - Mobile Reports (IDT)  191,494  3.60%
  4  2022  FTC - "877 FTC HELP" (Fraud & Other)  155,803  2.93%
  5  2022  FTC - "877 ID THEFT"  100,294  1.89%
  6  2023  FTC - Web Reports (IDT)  696,163  12.55%
  7  2023  FTC - Web Reports (Fraud & Other)  702,598  12.66%
  ...  ...  ...  ...  ...
  11  2023  FTC - "877 ID THEFT"  106,112  1.91%
  12  2024  FTC - Web Reports (IDT)  787,763  12.17%
  13  2024  FTC - Web Reports (Fraud & Other)  715,248  11.05%
  14  2024  FTC - Mobile Reports (Fraud & Other)  681,383  10.53%
  15  2024  FTC - Mobile Reports (IDT)  240,869  3.72%
  16  2024  FTC - "877 FTC HELP" (Fraud & Other)  135,095  2.09%
  17  2024  FTC - "877 ID THEFT"  104,102  1.61%

### Operator `ftc_fraud_web` (DataProcessing, executed)
Summary: Filter FTC data for "Fraud & Other" reports over web between 2022-2024: Filter rows containing "Fraud & Other" in Data Contributor column; filter for years 2022, 2023, 2024; keep all matching rows
Properties:
  code: def process(clean_ftc) -> pd.DataFrame:
    # Filter for rows containing 'Fraud & Other' in Data Contributor column
    df = clean_ftc[clean_ftc['Data Contributor'].str.contains('Fraud & Other')]
    # Filter for years 2022-2024
    df = df[df['Year'].astype(int).between(2022, 2024)]
    return df
Result:
  Executed operator ftc_fraud_web
  Input operator(table shape): clean_ftc(18, undefined)
  Output table shape: (9, 4)
    Year  Data Contributor  # of Reports  %
  0  2022  FTC - Web Reports (Fraud & Other)  693,789  13.05%
  1  2022  FTC - Mobile Reports (Fraud & Other)  622,035  11.70%
  2  2022  FTC - "877 FTC HELP" (Fraud & Other)  155,803  2.93%
  3  2023  FTC - Web Reports (Fraud & Other)  702,598  12.66%
  4  2023  FTC - Mobile Reports (Fraud & Other)  633,956  11.43%
  5  2023  FTC - "877 FTC HELP" (Fraud & Other)  137,005  2.47%
  6  2024  FTC - Web Reports (Fraud & Other)  715,248  11.05%
  7  2024  FTC - Mobile Reports (Fraud & Other)  681,383  10.53%
  8  2024  FTC - "877 FTC HELP" (Fraud & Other)  135,095  2.09%

### Operator `total_fraud_web` (DataProcessing, executed)
Summary: Sum the "# of Reports" column from ftc_fraud_web operator to get the total number of frauds reported by FTC over the web between 2022 and 2024
Properties:
  code: def process(ftc_fraud_web) -> pd.DataFrame:
    # Sum the '# of Reports' column to get the total number of frauds reported by FTC over the web between 2022-2024
    total = ftc_fraud_web['# of Reports'].str.replace(',', '').astype(int).sum()
    return pd.DataFrame({'Total Fraud Reports': [total]})

Result:
  Executed operator total_fraud_web
  Input operator(table shape): ftc_fraud_web(9, undefined)
  Output table shape: (1, 1)
    Total Fraud Reports
  0  4476912

## Links
- contributors → clean_ftc
- clean_ftc → ftc_fraud_web
- ftc_fraud_web → total_fraud_web'''.strip()

message = run_prompt(PROMPT)

Thought: The workflow has successfully processed the data and calculated the total number of frauds reported by FTC over the web between 2022 and 2024. The final result is 4,476,912 fraud reports. The workflow involved loading the data, cleaning and filtering it, and then summing the relevant reports. The dataflow is complete and the result is accurate.**Final Answer: 4476912**


## Multi-turn / Tool-result Pattern

如果你真的执行了工具，把 assistant 的上一次输出和工具 observation 继续放回 messages 里。这个 pattern 用于服务化后的多轮测试。

In [ ]:
messages = [{'role': 'user', 'content': PROMPT}]

# First model turn.
assistant_message = run_messages(messages, max_tokens=2048)

# If your harness executes a tool, append the model output and the observation,
# then ask the model to continue. Keep content text-compatible with the original
# ReACT format used in infer_prompt_swift_agent_bw.ipynb.
#
# messages.extend([
#     {'role': 'assistant', 'content': assistant_message.content or ''},
#     {'role': 'user', 'content': 'Observation:\n<tool execution result>\n'},
# ])
# next_message = run_messages(messages, max_tokens=2048)